# 🐱‍🐉 Clasificación de Pokémon con Transfer Learning (Explicado)

Este cuaderno es una guía paso a paso para la **Actividad 3**. 

**Objetivo:** Utilizar un modelo pre-entrenado (`EfficientNet`) para clasificar 150 tipos diferentes de Pokémon con alta precisión.

### 1. Configuración del Entorno
Antes de cocinar, necesitamos preparar la cocina. Aquí importamos las librerías necesarias (PyTorch) y seleccionamos el hardware más rápido disponible (GPU/CUDA o CPU).

In [ ]:
import torch
import torchvision
from torch import nn
from torchvision import transforms
import matplotlib.pyplot as plt

# Intentamos importar torchinfo para ver resúmenes del modelo
try:
    from torchinfo import summary
except:
    print("[INFO] Instalando torchinfo...")
    !pip install -q torchinfo
    from torchinfo import summary

# Configuración de dispositivo (Device Agnostic)
# Esto asegura que el código corra rápido en Nvidia (cuda) o Mac (mps)
device = "cuda" if torch.cuda.is_available() else "cpu"
device = "mps" if torch.backends.mps.is_available() else device

print(f"[INFO] Usando dispositivo: {device}")

### 2. Preparación de los Datos (El "Chef")

El modelo `EfficientNet` fue entrenado con millones de imágenes de una forma específica. Para que funcione bien con nuestros Pokémon, debemos "cocinar" (preprocesar) nuestras fotos exactamente igual.

* **Transformaciones:** Ajustar tamaño, normalizar colores.
* **DataLoaders:** Empaquetar las fotos en lotes (batches) de 32 para que la memoria no explote.

In [ ]:
import os
from torchvision import datasets
from torch.utils.data import DataLoader

# Rutas a los datos (Asegúrate de que estas carpetas existan)
train_dir = "../data/train/"
test_dir = "../data/test/"

# 1. Obtener los pesos (el conocimiento previo) de EfficientNet
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

# 2. Obtener las transformaciones automáticas que requiere este modelo
auto_transforms = weights.transforms()

# 3. Función para crear los cargadores de datos
def create_dataloaders(train_dir, test_dir, transform, batch_size=32):
    # Cargar carpetas como datasets
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    test_data = datasets.ImageFolder(test_dir, transform=transform)
    
    # Obtener nombres de las clases (ej: 'Pikachu', 'Bulbasaur')
    class_names = train_data.classes

    # Crear DataLoaders (lotes)
    train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_dataloader, test_dataloader, class_names

# Crear los dataloaders
train_dataloader, test_dataloader, class_names = create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=auto_transforms
)

print(f"[INFO] Clases encontradas: {len(class_names)}")
print(f"[INFO] Ejemplo de clases: {class_names[:5]}")

### 3. Construcción del Modelo (El "Arquitecto")

Esta es la parte clave del **Transfer Learning**. 

Cargamos un "cerebro" que ya sabe ver (`EfficientNet`) y le cambiamos la "boca" (la capa de salida) para que hable de Pokémon en lugar de objetos genéricos.

1.  **Descargar modelo:** Traemos la red neuronal ya entrenada.
2.  **Congelar base:** Bloqueamos las capas de visión (`features`) para no dañar lo que ya saben.
3.  **Cambiar cabeza:** Ponemos una capa nueva al final que tenga 150 salidas (una por cada Pokémon).

In [ ]:
# 1. Cargar el modelo base pre-entrenado
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

# 2. Congelar las capas base (Feature Extractor)
# Le decimos a PyTorch: "No calcules gradientes para esto, déjalo como está"
for param in model.features.parameters():
    param.requires_grad = False

# 3. Reemplazar la cabeza del clasificador (Classifier Head)
output_shape = len(class_names)

# Semillas para reproducibilidad
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Definir la nueva capa de salida
model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True), 
    torch.nn.Linear(in_features=1280, # 1280 es la salida fija de EfficientNetB0
                    out_features=output_shape, # 150 clases de Pokémon
                    bias=True)
).to(device)

# Ver resumen del modelo
summary(model, 
        input_size=(32, 3, 224, 224), 
        col_names=["input_size", "output_size", "num_params", "trainable"])

### 4. Entrenamiento (El "Entrenador")

Aquí definimos las reglas del juego:
* **Loss Function:** `CrossEntropyLoss` (Mide qué tan mal se equivocó el modelo).
* **Optimizer:** `Adam` (Ajusta los pesos para reducir el error).
* **Bucle:** Repetimos el proceso de estudiar (train) y hacer examen (test) varias veces.

In [ ]:
from tqdm.auto import tqdm
import time

# Definir pérdida y optimizador
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --- Funciones del Motor de Entrenamiento ---

def train_step(model, dataloader, loss_fn, optimizer, device):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)
    return train_loss / len(dataloader), train_acc / len(dataloader)

def test_step(model, dataloader, loss_fn, device):
    model.eval()
    test_loss, test_acc = 0, 0
    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            test_pred_logits = model(X)
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))
    return test_loss / len(dataloader), test_acc / len(dataloader)

def train(model, train_dataloader, test_dataloader, optimizer, loss_fn, epochs, device):
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model, train_dataloader, loss_fn, optimizer, device)
        test_loss, test_acc = test_step(model, test_dataloader, loss_fn, device)
        print(f"Epoch: {epoch+1} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
    return results

### 5. ¡Ejecutar el Entrenamiento!
Ahora unimos todo. Si todo sale bien, verás cómo la precisión (`Test Acc`) sube rápidamente por encima del 80%.

In [ ]:
# Ejecutar por 5 o 10 épocas
start_time = time.time()

results = train(model=model, 
                train_dataloader=train_dataloader, 
                test_dataloader=test_dataloader, 
                optimizer=optimizer, 
                loss_fn=loss_fn, 
                epochs=10, 
                device=device)

print(f"[INFO] Tiempo total de entrenamiento: {time.time()-start_time:.2f} segundos")